# Google MLCC: standalone Colab edition

Run the setup cell in a fresh Colab runtime, then run every cell from top to bottom. Setup downloads the original course data and installs compatible dependencies, including `google-ml-edu` (imported as `ml_edu`) for taxi/rice. No course ZIP is required. TensorFlow 2.21 and Keras 3.12+ adapt those exercises to current Python 3.13 Colab; the original TF 2.18/NumPy 2.0 pins do not support Python 3.13.

`USE_GOOGLE_DRIVE=True` saves measured results, models, and task execution evidence to `MyDrive/HaiLong_Colab_Results/01_google_mlcc`. Save this notebook in Drive and save it after running to retain cell outputs. The completion report requires every computational stage and full original data/epochs. Website quizzes and interactive exercises have separate progress.

Original source: https://raw.githubusercontent.com/google/eng-edu/main/ml/cc/exercises/linear_regression_taxi.ipynb


In [ ]:
# Standalone Google MLCC setup. Start a fresh Colab runtime, then Run all.
# No ZIP or eng-edu clone is required. Full original data/tasks only.
from pathlib import Path
import os, sys, subprocess, importlib.metadata as metadata
import hashlib, json, shutil, urllib.request, platform, uuid
from datetime import datetime, timezone
from packaging.requirements import Requirement

USE_GOOGLE_DRIVE = True  #@param {type:"boolean"}
QUICK_RUN = False  # retained for compatibility; full completion requires False
if QUICK_RUN:
    raise ValueError('Use QUICK_RUN=False to execute the full original exercise.')
NOTEBOOK_NAME = '01_linear_regression_taxi.ipynb'
DEPENDENCY_PROFILE = 'google'
REQUIREMENTS = ['google-ml-edu==0.1.3', 'tensorflow==2.21.0', 'keras>=3.12,<4', 'numpy>=2,<3', 'pandas>=2.2,<3', 'matplotlib>=3.9,<4', 'plotly>=5,<7', 'scikit-learn>=1.5,<2']
REQUIRED_TASKS = ['describe_data', 'correlations_and_visualization', 'single_feature_training', 'three_hyperparameter_experiments', 'two_feature_training', 'compare_models_and_units', 'predict_fares']
DATA_SOURCES = [{'filename': 'chicago_taxi_train.csv', 'url': 'https://download.mlcc.google.com/mledu-datasets/chicago_taxi_train.csv', 'sha256': '1aa03dc1c10909223bfc41ff6a90f93f1fd5ba6f64959c0aa1b94dfa94ae76d4', 'rows': 31694}]
ORIGINAL_SOURCE = 'https://raw.githubusercontent.com/google/eng-edu/main/ml/cc/exercises/linear_regression_taxi.ipynb'
VISUALIZATION_ADAPTATION = None

try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

# TensorFlow 2.21 has Python 3.13 wheels; original TF 2.18 pins do not.
# Set Keras mode before importing TensorFlow or Keras.
if DEPENDENCY_PROFILE in ('google', 'fairness'):
    if DEPENDENCY_PROFILE == 'fairness':
        if any(n in sys.modules for n in ('tensorflow', 'keras', 'tf_keras')):
            if os.environ.get('TF_USE_LEGACY_KERAS') != '1':
                raise RuntimeError('Select Runtime > Disconnect and delete runtime, reconnect, then Run all: MinDiff needs legacy Keras selected before TensorFlow import.')
        os.environ['TF_USE_LEGACY_KERAS'] = '1'
    else:
        if os.environ.get('TF_USE_LEGACY_KERAS') == '1' and any(n in sys.modules for n in ('tensorflow', 'keras', 'tf_keras')):
            raise RuntimeError('Start a fresh runtime after fairness before running this Keras 3 notebook.')
        os.environ.pop('TF_USE_LEGACY_KERAS', None)
    os.environ['KERAS_BACKEND'] = 'tensorflow'
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'
os.environ['COURSE_SMOKE'] = '0'
os.environ['WANDB_MODE'] = 'offline'

# google-ml-edu is the distribution that provides the ml_edu import.
IMPORT_NAMES = {'google-ml-edu':'ml_edu', 'tf-keras':'tf_keras',
    'tensorflow-model-remediation':'tensorflow_model_remediation',
    'scikit-learn':'sklearn'}
def installed_version(package):
    try:
        return metadata.version(package)
    except metadata.PackageNotFoundError:
        return None
missing, loaded_changes = [], []
for text in REQUIREMENTS:
    req = Requirement(text)
    version = installed_version(req.name)
    if version is None or version not in req.specifier:
        missing.append(text)
        module = IMPORT_NAMES.get(req.name, req.name.replace('-', '_'))
        if module in sys.modules:
            loaded_changes.append(module)
if missing and IN_COLAB:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--disable-pip-version-check', *missing])
    if loaded_changes:
        raise RuntimeError('Installed required versions. Select Runtime > Restart session, then Run all again; old loaded modules: ' + ', '.join(loaded_changes))
elif missing:
    raise RuntimeError('Prepared for Colab. Missing compatible local packages: ' + ', '.join(missing))

if IN_COLAB:
    COURSE_ROOT = Path('/content/HaiLong_Colab_Exercises')
else:
    COURSE_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
                        if (p / 'colab_runtime.py').is_file()), None)
    if COURSE_ROOT is None:
        raise RuntimeError('Run in Colab or inside HaiLong_Colab_Exercises.')
COURSE_DIR = COURSE_ROOT / '01_google_mlcc'
COURSE_DIR.mkdir(parents=True, exist_ok=True)
os.chdir(COURSE_DIR)
DATA = COURSE_DIR / 'data'
DATA.mkdir(exist_ok=True)
DRIVE_DIR = None
if IN_COLAB and USE_GOOGLE_DRIVE:
    if not Path('/content/drive/MyDrive').is_dir():
        drive.mount('/content/drive', timeout_ms=300000)
    DRIVE_DIR = Path('/content/drive/MyDrive/HaiLong_Colab_Results/01_google_mlcc')
    DRIVE_DIR.mkdir(parents=True, exist_ok=True)
for folder in ('results', 'models'):
    target = COURSE_DIR / folder
    if DRIVE_DIR is not None:
        (DRIVE_DIR / folder).mkdir(exist_ok=True)
        if not target.exists():
            target.symlink_to(DRIVE_DIR / folder, target_is_directory=True)
    target.mkdir(exist_ok=True)
RESULTS, MODELS = COURSE_DIR / 'results', COURSE_DIR / 'models'
for spec in DATA_SOURCES:
    target = DATA / spec['filename']
    if not target.is_file():
        temporary = target.with_suffix(target.suffix + '.download')
        print('Downloading original Google course data:', spec['url'])
        urllib.request.urlretrieve(spec['url'], temporary)
        if hashlib.sha256(temporary.read_bytes()).hexdigest() != spec['sha256']:
            raise ValueError('Downloaded dataset differs from verified original: ' + spec['filename'])
        temporary.replace(target)
    if hashlib.sha256(target.read_bytes()).hexdigest() != spec['sha256']:
        raise ValueError('Existing data differs from the original course dataset: ' + str(target))

EXECUTED_TASKS = {}
COLAB_CONTEXT = {'notebook': NOTEBOOK_NAME, 'run_id': str(uuid.uuid4()),
    'started_at_utc': datetime.now(timezone.utc).isoformat(),
    'in_hosted_colab': IN_COLAB, 'quick_run': False,
    'python': platform.python_version(),
    'versions': {Requirement(r).name: installed_version(Requirement(r).name) for r in REQUIREMENTS},
    'original_source': ORIGINAL_SOURCE, 'data_sources': DATA_SOURCES,
    'visualization_adaptation': VISUALIZATION_ADAPTATION}
def mark_task(name, **evidence):
    if name not in REQUIRED_TASKS:
        raise KeyError(name)
    EXECUTED_TASKS[name] = {'recorded_at_utc': datetime.now(timezone.utc).isoformat(), **evidence}
    checkpoint = {**COLAB_CONTEXT, 'last_completed_stage': name,
        'updated_at_utc': datetime.now(timezone.utc).isoformat(),
        'missing_tasks': sorted(set(REQUIRED_TASKS) - set(EXECUTED_TASKS)),
        'task_evidence': EXECUTED_TASKS, 'course_website_completion_confirmed': False}
    (RESULTS / (Path(NOTEBOOK_NAME).stem + '_progress.json')).write_text(
        json.dumps(checkpoint, indent=2, allow_nan=False), encoding='utf-8')
    if DRIVE_DIR is not None:
        for folder in ('results', 'models'):
            source, target = COURSE_DIR / folder, DRIVE_DIR / folder
            if source.resolve() != target.resolve():
                shutil.copytree(source, target, dirs_exist_ok=True)
def finish_run(context):
    missing_tasks = sorted(set(REQUIRED_TASKS) - set(EXECUTED_TASKS))
    if missing_tasks:
        raise RuntimeError('Required exercise stages did not run: ' + ', '.join(missing_tasks))
    report = {**context, 'finished_at_utc': datetime.now(timezone.utc).isoformat(),
        'full_original_computational_tasks_executed': True,
        'course_website_completion_confirmed': False,
        'required_tasks': REQUIRED_TASKS, 'task_evidence': EXECUTED_TASKS,
        'note': 'Execution evidence covers this notebook. It does not certify website quizzes, interactive exercises, or personal reflection.'}
    path = RESULTS / (Path(NOTEBOOK_NAME).stem + '_completion.json')
    path.write_text(json.dumps(report, indent=2, allow_nan=False), encoding='utf-8')
    if DRIVE_DIR is not None:
        for folder in ('results', 'models'):
            source, target = COURSE_DIR / folder, DRIVE_DIR / folder
            if source.resolve() != target.resolve():
                shutil.copytree(source, target, dirs_exist_ok=True)
        print('Saved computational progress and artifacts to:', DRIVE_DIR)
    print('Full original computational stages executed:', NOTEBOOK_NAME)
    print('Completion evidence:', path)
    return report
print('Setup ready:', NOTEBOOK_NAME, '| full course data |', DEPENDENCY_PROFILE)

# Completed Google MLCC: taxi regression
Source: https://github.com/google/eng-edu/blob/main/ml/cc/exercises/linear_regression_taxi.ipynb
Google's Apache-2.0 teaching notebook, with all three hyperparameter trials implemented and a held-out evaluation extension. Instructor answers remain labelled as instructor examples; use the recorded outputs for this run.


In [ ]:
#@title Copyright 2023 Google LLC. Double-click here for license information.
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
# https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Colabs

Machine Learning Crash Course uses Colaboratories (Colabs) for all programming exercises. Colab is Google's implementation of [Jupyter Notebook](https://jupyter.org/). For more information about Colabs and how to use them, go to [Welcome to Colaboratory](https://research.google.com/colaboratory).

# Linear Regression
In this Colab you will use a real dataset to train a model to predict the fare of a taxi ride in Chicago, Illinois.

## Learning Objectives
After completing this Colab, you'll be able to:

  * Read a .csv file into a [pandas](https://developers.google.com/machine-learning/glossary/#pandas) DataFrame.
  * Explore a [dataset](https://developers.google.com/machine-learning/glossary/#data_set) with Python visualization libraries.
  * Experiment with different [features](https://developers.google.com/machine-learning/glossary/#feature) to build a linear regression model.
  * Tune the model's [hyperparameters](https://developers.google.com/machine-learning/glossary/#hyperparameter).
  * Compare training runs using [root mean squared error](https://developers.google.com/machine-learning/glossary/#root-mean-squared-error-rmse) and [loss curves](https://developers.google.com/machine-learning/glossary/#loss-curve).

## Dataset Description
The [dataset for this exercise](https://storage.mtls.cloud.google.com/mlcc-nextgen-internal/chicago_taxi_train.csv) is derived from the [City of Chicago Taxi Trips dataset](https://data.cityofchicago.org/Transportation/Taxi-Trips/wrvz-psew). The data for this exercise is a subset of the Taxi Trips data, and focuses on a two-day period in May of 2022.

# Part 1 - Setup Exercise


---

## Load required modules

This exercise depends on several Python libraries to help with data manipulation, machine learning tasks, and data visualization.

**Instructions**
1. Run the **Install required libraries** code cell (below).
1. Run the **Load dependencies** code cell (below).

In [ ]:
from pathlib import Path
import os
os.environ.setdefault('KERAS_BACKEND', 'tensorflow')
os.environ.setdefault('TF_CPP_MIN_LOG_LEVEL', '3')
os.environ.setdefault('TF_NUM_INTRAOP_THREADS', '1')
os.environ.setdefault('TF_NUM_INTEROP_THREADS', '1')
# Colab setup above selects this course folder.
DATA = Path('data')
RESULTS = Path('results'); RESULTS.mkdir(exist_ok=True)

# Dependencies are installed by the Colab setup cell.

In [ ]:
#@title Code - Load dependencies

# data
import numpy as np
import pandas as pd

# machine learning
import keras
import ml_edu.experiment
import ml_edu.results

# data visualization
import plotly.express as px

## Load the dataset


The following code cell loads the dataset and creates a pandas DataFrame.

You can think of a DataFrame like a spreadsheet with rows and columns. The rows represent individual data examples, and the columns represent the attributes associated with each example.

In [ ]:
chicago_taxi_dataset = pd.read_csv(DATA / 'chicago_taxi_train.csv')

assert len(chicago_taxi_dataset) == 31694, 'Use the complete original taxi dataset.'

## Update the dataframe

The following code cell updates the DataFrame to use only specific columns from the dataset.

Notice that that output shows just a sample of the dataset, but there should be enough information for you to identify the features associated with the dataset, and have a look at the actual data for a few examples.

In [ ]:
#@title Code - Read dataset

# Updates dataframe to use specific columns.
training_df = chicago_taxi_dataset.loc[:, ('TRIP_MILES', 'TRIP_SECONDS', 'FARE', 'COMPANY', 'PAYMENT_TYPE', 'TIP_RATE')]

print('Read dataset completed successfully.')
print('Total number of rows: {0}\n\n'.format(len(training_df.index)))
training_df.head(200)

# Part 2 - Dataset Exploration


---

## View dataset statistics

A large part of most machine learning projects is getting to know your data. In this step, you will use the ``DataFrame.describe`` method to view descriptive statistics about the dataset and answer some important questions about the data.

**Instructions**
1. Run the **View dataset statistics** code cell.
1. Inspect the output and answer these questions:
  * What is the maximum fare?
  * What is the mean distance across all trips?
  * How many cab companies are in the dataset?
  * What is the most frequent payment type?
  * Are any features missing data?
1. Run the code **View answers to dataset statistics** code cell to check your answers.


You might be wondering why there are groups of `NaN` (not a number) values listed in the output. When working with data in Python, you may see this value if the result of a calculation can not be computed or if there is missing information. For example, in the taxi dataset `PAYMENT_TYPE` and `COMPANY` are non-numeric, categorical features; numeric information such as mean and max do not make sense for categorical features so the output displays `NaN`.

In [ ]:
display(training_df.describe())
training_df.describe().to_csv(RESULTS / 'taxi_statistics.csv')
mark_task('describe_data', rows=len(training_df))

In [ ]:
#@title Double-click or run to view answers about dataset statistics

answer = '''
What is the maximum fare? 				              Answer: $159.25
What is the mean distance across all trips? 		Answer: 8.2895 miles
How many cab companies are in the dataset? 		  Answer: 31
What is the most frequent payment type? 		    Answer: Credit Card
Are any features missing data? 				          Answer: No
'''

# You should be able to find the answers to the questions about the dataset
# by inspecting the table output after running the DataFrame describe method.
#
# Run this code cell to verify your answers.

# What is the maximum fare?
max_fare = training_df['FARE'].max()
print("What is the maximum fare? 				Answer: ${fare:.2f}".format(fare = max_fare))

# What is the mean distance across all trips?
mean_distance = training_df['TRIP_MILES'].mean()
print("What is the mean distance across all trips? 		Answer: {mean:.4f} miles".format(mean = mean_distance))

# How many cab companies are in the dataset?
num_unique_companies =  training_df['COMPANY'].nunique()
print("How many cab companies are in the dataset? 		Answer: {number}".format(number = num_unique_companies))

# What is the most frequent payment type?
most_freq_payment_type = training_df['PAYMENT_TYPE'].value_counts().idxmax()
print("What is the most frequent payment type? 		Answer: {type}".format(type = most_freq_payment_type))

# Are any features missing data?
missing_values = training_df.isnull().sum().sum()
print("Are any features missing data? 				Answer:", "No" if missing_values == 0 else "Yes")

## Generate a correlation matrix

An important part of machine learning is determining which [features](https://developers.google.com/machine-learning/glossary/#feature) correlate with the [label](https://developers.google.com/machine-learning/glossary/#label). If you have ever taken a taxi ride before, your experience is probably telling you that the fare is typically associated with the distance traveled and the duration of the trip. But, is there a way for you to learn more about how well these features correlate to the fare (label)?

In this step, you will use a **correlation matrix** to identify features whose values correlate well with the label. Correlation values have the following meanings:

  * **`1.0`**: perfect positive correlation; that is, when one attribute rises, the other attribute rises.
  * **`-1.0`**: perfect negative correlation; that is, when one attribute rises, the other attribute falls.
  * **`0.0`**: no correlation; the two columns [are not linearly related](https://en.wikipedia.org/wiki/Correlation_and_dependence#/media/File:Correlation_examples2.svg).

In general, the higher the absolute value of a correlation value, the greater its predictive power.

**Instructions**

1. Inspect the code in the **View correlation matrix** code cell.
1. Run the **View correlation matrix** code cell and inspect the output.
1. **Check your understanding** by answering these questions:
  * Which feature correlates most strongly to the label FARE?
  * Which feature correlates least strongly to the label FARE?


In [ ]:
#@title Code - View correlation matrix
training_df.corr(numeric_only = True)

In [ ]:
#@title Double-click to view answers about the correlation matrix

# Which feature correlates most strongly to the label FARE?
# ---------------------------------------------------------
answer = '''
The feature with the strongest correlation to the FARE is TRIP_MILES.
As you might expect, TRIP_MILES looks like a good feature to start with to train
the model. Also, notice that the feature TRIP_SECONDS has a strong correlation
with fare too.
'''
print(answer)


# Which feature correlates least strongly to the label FARE?
# -----------------------------------------------------------
answer = '''The feature with the weakest correlation to the FARE is TIP_RATE.'''
print(answer)

## Visualize relationships in dataset

Sometimes it is helpful to visualize relationships between features in a dataset; one way to do this is with a pair plot. A **pair plot** generates a grid of pairwise plots to visualize the relationship of each feature with all other features all in one place.

**Instructions**
1. Run the **View pair plot** code cell.

In [ ]:
#@title Code - View pairplot
px.scatter_matrix(training_df, dimensions=["FARE", "TRIP_MILES", "TRIP_SECONDS"]).show()

mark_task('correlations_and_visualization', features=list(training_df.columns))

# Part 3 - Train Model


---

## Define functions to build and train a model

The code you need to build and train your model is in the **Define ML functions** code cell. If you would like to explore this code, expand the code cell and take a look.

**Instructions**
1. Run the **Define ML functions** code cell.

In [ ]:
#@title Code - Define ML functions

def create_model(
    settings: ml_edu.experiment.ExperimentSettings,
    metrics: list[keras.metrics.Metric],
) -> keras.Model:
  """Create and compile a simple linear regression model."""
  # Describe the topography of the model.
  # The topography of a simple linear regression model
  # is a single node in a single layer.
  inputs = {name: keras.Input(shape=(1,), name=name) for name in settings.input_features}
  concatenated_inputs = keras.layers.Concatenate()(list(inputs.values()))
  outputs = keras.layers.Dense(units=1)(concatenated_inputs)
  model = keras.Model(inputs=inputs, outputs=outputs)

  # Compile the model topography into code that Keras can efficiently
  # execute. Configure training to minimize the model's mean squared error.
  model.compile(optimizer=keras.optimizers.RMSprop(learning_rate=settings.learning_rate),
                loss="mean_squared_error",
                metrics=metrics)

  return model


def train_model(
    experiment_name: str,
    model: keras.Model,
    dataset: pd.DataFrame,
    label_name: str,
    settings: ml_edu.experiment.ExperimentSettings,
) -> ml_edu.experiment.Experiment:
  """Train the model by feeding it data."""

  # Feed the model the feature and the label.
  # The model will train for the specified number of epochs.
  features = {name: dataset[name].values for name in settings.input_features}
  label = dataset[label_name].values
  history = model.fit(x=features,
                      y=label,
                      batch_size=settings.batch_size,
                      epochs=settings.number_epochs, verbose=0)

  return ml_edu.experiment.Experiment(
      name=experiment_name,
      settings=settings,
      model=model,
      epochs=history.epoch,
      metrics_history=pd.DataFrame(history.history),
  )

print("SUCCESS: defining linear regression functions complete.")

## Train a model with one feature

In this step you will train a model to predict the cost of the fare using a **single feature**. Earlier, you saw that `TRIP_MILES` (distance) correlates most strongly with the ``FARE``, so let's start with `TRIP_MILES` as the feature for your first training run.

**Instructions**

1. Run the **Experiment 1** code cell to build your model with one feature.
1. Review the output from the training run
1. **Check your understanding** by answering these questions:
  * How many epochs did it take to converge on the final model?
  * How well does the model fit the sample data?

During training, you should see the root mean square error (RMSE) in the output. The units for RMSE are the same as the units for the label (dollars). In other words, you can use the RMSE to determine how far off, on average, the predicted fares are in dollars from the observed values.

In [ ]:
#@title Code - Experiment 1

# The following variables are the hyperparameters.
settings_1 = ml_edu.experiment.ExperimentSettings(
    learning_rate = 0.001,
    number_epochs = 20,
    batch_size = 50,
    input_features = ['TRIP_MILES']
)

keras.utils.set_random_seed(42)
metrics = [keras.metrics.RootMeanSquaredError(name='rmse')]

model_1 = create_model(settings_1, metrics)

experiment_1 = train_model('one_feature', model_1, training_df, 'FARE', settings_1)

ml_edu.results.plot_experiment_metrics(experiment_1, ['rmse'])
ml_edu.results.plot_model_predictions(experiment_1, training_df, 'FARE')

assert len(experiment_1.epochs) == 20
experiment_1.metrics_history.to_csv(RESULTS / 'taxi_single_feature_history.csv', index=False)
experiment_1.model.save(MODELS / 'taxi_single_feature.keras')
rmse_curve = experiment_1.metrics_history['rmse']
best_epoch = int(rmse_curve.to_numpy().argmin()) + 1
print(f'Single-feature run: lowest observed training RMSE ${rmse_curve.min():.2f} at epoch {best_epoch}; final RMSE ${rmse_curve.iloc[-1]:.2f}. Inspect the curve to decide whether it has plateaued.')
mark_task('single_feature_training', epochs=len(experiment_1.epochs), final_rmse=float(rmse_curve.iloc[-1]), lowest_rmse_epoch=best_epoch)

In [ ]:
#@title Double-click to view answers for training model with one feature

# How many epochs did it take to converge on the final model?
# -----------------------------------------------------------------------------
answer = """
Use the loss curve to see where the loss begins to level off during training.

With this set of hyperparameters:

  learning_rate = 0.001
  epochs = 20
  batch_size = 50

it takes about 5 epochs for the training run to converge to the final model.
"""
print(answer)

# How well does the model fit the sample data?
# -----------------------------------------------------------------------------
answer = '''
It appears from the model plot that the model fits the sample data fairly well.
'''
print(answer)

## Experiment with hyperparameters

It is common with machine learning to run multiple experiments to find the best set of hyperparmeters to train your model. In this step, try varying the hyperparameters one by one with this set of experiments:

* *Experiment 1:* **Increase** the learning rate to **``1.0``** (batch size at ``50``).
* *Experiment 2:* **Decrease** the learning rate to **``0.0001``** (batch size at ``50``).
* *Experiment 3:* **Increase** the batch size to **``500``** (learning rate at ``0.001``).

**Instructions**
1. Update the hyperparameter values in the **Experiment 2** code cell according to the experiment.
2. Run the **Experiment 2** code cell.
3. After the training run, examine the output and note any differences you see in the loss curve or model output.
4. Repeat steps 1 - 3 for each hyperparameter experiment.
5. **Check your understanding** by answering these questions:
  * How did raising the learning rate impact your ability to train the model?
  * How did lowering the learning rate impact your ability to train the model?
  * Did changing the batch size effect your training results?


In [ ]:
# Complete all three requested hyperparameter experiments with the same seed.
hyper_experiments = []
for experiment_name, lr, bs in [('high_lr', 1.0, 50), ('low_lr', 0.0001, 50), ('large_batch', 0.001, 500)]:
    keras.utils.set_random_seed(42)
    settings_2 = ml_edu.experiment.ExperimentSettings(
        learning_rate=lr, number_epochs=20, batch_size=bs, input_features=['TRIP_MILES'])
    model_2 = create_model(settings_2, [keras.metrics.RootMeanSquaredError(name='rmse')])
    experiment_2 = train_model(experiment_name, model_2, training_df, 'FARE', settings_2)
    hyper_experiments.append(experiment_2)
    experiment_2.metrics_history.to_csv(RESULTS / f'taxi_{experiment_name}_history.csv', index=False)
    ml_edu.results.plot_experiment_metrics(experiment_2, ['rmse'])
hyper_results = pd.DataFrame([
    {'experiment': e.name, 'learning_rate': e.settings.learning_rate,
     'batch_size': e.settings.batch_size, 'final_rmse': e.get_final_metric_value('rmse')}
    for e in hyper_experiments])
print(hyper_results.to_string(index=False))
hyper_results.to_csv(RESULTS / 'taxi_hyperparameters.csv', index=False)

assert len(hyper_experiments) == 3 and all(len(e.epochs) == 20 for e in hyper_experiments)
baseline_rmse = float(experiment_1.get_final_metric_value('rmse'))
for row in hyper_results.itertuples():
    print(f'{row.experiment}: final RMSE ${row.final_rmse:.2f}; change from baseline ${row.final_rmse-baseline_rmse:+.2f}. Compare the learning curves and final values.')
mark_task('three_hyperparameter_experiments', settings=hyper_results.to_dict(orient='records'), epochs_each=20)

In [ ]:
#@title Double-click to view answers for hyperparameter experiments

# How did raising the learning rate impact your ability to train the model?
# -----------------------------------------------------------------------------
answer = """
When the learning rate is too high, the loss curve bounces around and does not
appear to be moving towards convergence with each iteration. Also, notice that
the predicted model does not fit the data very well. With a learning rate that
is too high, it is unlikely that you will be able to train a model with good
results.
"""
print(answer)

# How did lowering the learning rate impact your ability to train the model?
# -----------------------------------------------------------------------------
answer = '''
When the learning rate is too small, it may take longer for the loss curve to
converge. With a small learning rate the loss curve decreases slowly, but does
not show a dramatic drop or leveling off. With a small learning rate you could
increase the number of epochs so that your model will eventually converge, but
it will take longer.
'''
print(answer)

# Did changing the batch size effect your training results?
# -----------------------------------------------------------------------------
answer = '''
Increasing the batch size makes each epoch run faster, but as with the smaller
learning rate, the model does not converge with just 20 epochs. If you have
time, try increasing the number of epochs and eventually you should see the
model converge.
'''
print(answer)

## Train a model with two features

The model you trained with the feature ``TOTAL_MILES`` demonstrates fairly strong predictive power, but is it possible to do better? In this step, try training the model with two features, ``TRIP_MILES`` and ``TRIP_MINUTES``, to see if you can improve the model. You may recall that the original dataset does not include a feature ``TRIP_MINUTES``, but this feature can be easily derived from ``TRIP_SECONDS`` as shown in the code below.*

**Instructions**
1. Review the code in **Experiment 3** code cell.
1. Run the **Experiment 3** code cell.
1. Review the output from the training run and answer these questions:
  * Does the model with two features produce better results than one using a single feature?
  * Does it make a difference if you use ``TRIP_SECONDS`` instead of ``TRIP_MINUTES``?
  * How well do you think the model comes to the ground truth fare calculation for Chicago Taxi Trips?


Notice that the scatter plot of the features vs. the label is a three dimensional (3-D) plot. This representation allows you to visualize both features and the label all together. The two features (TRIP_MILES and TRIP_MINUTES) are on the x and y axis, and the label (FARE) is on the z axis. The plot shows individual examples in the dataset as circles, and the model as a surface (plane). With this 3-D model, if the trained model is good you would expect most of the examples to land on the plane surface. The 3-D plot is interactive so you can explore the data further by clicking or dragging the plot.

In [ ]:
#@title Code - Experiment 3

# The following variables are the hyperparameters.
settings_3 = ml_edu.experiment.ExperimentSettings(
    learning_rate = 0.001,
    number_epochs = 20,
    batch_size = 50,
    input_features = ['TRIP_MILES', 'TRIP_MINUTES']
)

training_df['TRIP_MINUTES'] = training_df['TRIP_SECONDS']/60

keras.utils.set_random_seed(42)
metrics = [keras.metrics.RootMeanSquaredError(name='rmse')]

model_3 = create_model(settings_3, metrics)

experiment_3 = train_model('two_features', model_3, training_df, 'FARE', settings_3)

ml_edu.results.plot_experiment_metrics(experiment_3, ['rmse'])
ml_edu.results.plot_model_predictions(experiment_3, training_df, 'FARE')

assert len(experiment_3.epochs) == 20
experiment_3.metrics_history.to_csv(RESULTS / 'taxi_two_feature_minutes_history.csv', index=False)
experiment_3.model.save(MODELS / 'taxi_two_feature_minutes.keras')
mark_task('two_feature_training', epochs=len(experiment_3.epochs), features=list(settings_3.input_features), final_rmse=float(experiment_3.get_final_metric_value('rmse')))

In [ ]:
#@title Double-click to view answers for training with two features

# Does the model with two features produce better results than one using a
# single feature?
# -----------------------------------------------------------------------------
answer = '''
To answer this question for your specific training runs, compare the RMSE for
each model. For example, if the RMSE for the model trained with one feature was
3.7457 and the RMSE for the model with two features is 3.4787, that means that
on average the model with two features makes predictions that are about $0.27
closer to the observed fare.

'''
print(answer)

# Does it make a difference if you use TRIP_SECONDS instead of TRIP_MINUTES?
# -----------------------------------------------------------------------------
answer = '''
When training a model with more than one feature, it is important that all
numeric values are roughly on the same scale. In this case, TRIP_SECONDS and
TRIP_MILES do not meet this criteria. The mean value for TRIP_MILES is 8.3 and
the mean for TRIP_SECONDS is 1,320; that is two orders of magnitude difference.
In contrast, the mean for TRIP_MINUTES is 22, which is more similar to the scale
of TRIP_MILES (8.3) than TRIP_SECONDS (1,320). Of course, this is not the
only way to scale values before training, but you will learn about that in
another module.
'''
print(answer)

# How well do you think the model comes to the ground truth fare calculation for
# Chicago taxi trips?
# -----------------------------------------------------------------------------
answer = '''
In reality, Chicago taxi cabs use a documented formula to determine cab fares.
For a single passenger paying cash, the fare is calculated like this:

FARE = 2.25 * TRIP_MILES + 0.12 * TRIP_MINUTES + 3.25

Typically with machine learning problems you would not know the 'correct'
formula, but in this case you can use this knowledge to evaluate your model.
Take a look at your model output (the weights and bias) and determine how
well it matches the ground truth fare calculation. You should find that the
model is roughly close to this formula.
'''
print(answer)

## Compare Experiments

In [ ]:
ml_edu.results.compare_experiment([experiment_1, experiment_3], ['rmse'], training_df, training_df['FARE'].values)

# Measure the original question about using seconds instead of minutes.
# Unit changes alter optimizer scaling even though they contain the same information.
keras.utils.set_random_seed(42)
seconds_settings = ml_edu.experiment.ExperimentSettings(
    learning_rate=0.001, number_epochs=20, batch_size=50,
    input_features=['TRIP_MILES', 'TRIP_SECONDS'])
seconds_model = create_model(seconds_settings, [keras.metrics.RootMeanSquaredError(name='rmse')])
seconds_experiment = train_model('two_features_seconds', seconds_model, training_df, 'FARE', seconds_settings)
assert len(seconds_experiment.epochs) == 20
seconds_experiment.metrics_history.to_csv(RESULTS / 'taxi_two_feature_seconds_history.csv', index=False)
comparison = pd.DataFrame([{'experiment': e.name,
    'features': ', '.join(e.settings.input_features),
    'final_training_rmse': float(e.get_final_metric_value('rmse'))}
    for e in [experiment_1, experiment_3, seconds_experiment]])
display(comparison)
comparison.to_csv(RESULTS / 'taxi_model_comparison.csv', index=False)
print('Two features minus one feature, training RMSE ($):', float(experiment_3.get_final_metric_value('rmse')-experiment_1.get_final_metric_value('rmse')))
print('Seconds minus minutes, training RMSE ($):', float(seconds_experiment.get_final_metric_value('rmse')-experiment_3.get_final_metric_value('rmse')))
print('Seconds and minutes contain the same information. Their scaling can change convergence with the same optimizer settings. Training fit alone cannot establish the true fare formula or unseen-trip accuracy.')
mark_task('compare_models_and_units', comparison=comparison.to_dict(orient='records'), seconds_epochs=20)

# Part 4 - Validate Model


---

## Use the model to make predictions

Now that you have a trained model, you can use the model to make predictions. In practice, you should make predictions on examples that are not used during training. However, for this exercise, you'll just work with a subset of the same training dataset. In another Colab exercise you will explore ways to make predictions on examples not used in training.

**Instructions**

1. Run the **Define functions to make predictions** code cell.
1. Run the **Make predictions** code cell.
1. Review the predictions in the output.
1. **Check your understanding** by answering these questions:
  * How close is the predicted value to the label value? In other words, does your model accurately predict the fare for a taxi ride?

In [ ]:
#@title Code - Define functions to make predictions
def format_currency(x):
  return "${:.2f}".format(x)

def build_batch(df, batch_size):
  batch = df.sample(n=batch_size).copy()
  batch.set_index(np.arange(batch_size), inplace=True)
  return batch

def predict_fare(model, df, features, label, batch_size=50):
  batch = build_batch(df, batch_size)
  predicted_values = model.predict_on_batch(x={name: batch[name].values for name in features})

  data = {"PREDICTED_FARE": [], "OBSERVED_FARE": [], "L1_LOSS": [],
          features[0]: [], features[1]: []}
  for i in range(batch_size):
    predicted = predicted_values[i][0]
    observed = batch.at[i, label]
    data["PREDICTED_FARE"].append(format_currency(predicted))
    data["OBSERVED_FARE"].append(format_currency(observed))
    data["L1_LOSS"].append(format_currency(abs(observed - predicted)))
    data[features[0]].append(batch.at[i, features[0]])
    data[features[1]].append("{:.2f}".format(batch.at[i, features[1]]))

  output_df = pd.DataFrame(data)
  return output_df

def show_predictions(output):
  header = "-" * 80
  banner = header + "\n" + "|" + "PREDICTIONS".center(78) + "|" + "\n" + header
  print(banner)
  print(output)
  return

In [ ]:
#@title Code - Make predictions

output = predict_fare(experiment_3.model, training_df, experiment_3.settings.input_features, 'FARE')
show_predictions(output)

output.to_csv(RESULTS / 'taxi_original_task_predictions.csv', index=False)
assert len(output) == 50
print('These 50 predictions use training examples, as the original exercise requests. The held-out extension below estimates performance on unseen rows.')
mark_task('predict_fares', prediction_rows=len(output))

In [ ]:
#@title Double-click to view answers for validate model

# How close is the predicted value to the label value?
# -----------------------------------------------------------------------------
answer = '''
Based on a random sampling of examples, the model seems to do pretty well
predicting the fare for a taxi ride. Most of the predicted values do not vary
significantly from the observed value. You should be able to see this by looking
at the column L1_LOSS = |observed - predicted|.
'''
print(answer)

## Research extension: genuine held-out evaluation
The original final section samples training rows. It demonstrates predictions but is not an independent generalisation test. This extension fits scaling on training rows and evaluates an untouched test split. The fare formula earlier is the course's historical teaching reference, not a current fare quote.


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import root_mean_squared_error
train_i, test_i = train_test_split(np.arange(len(training_df)), test_size=.2, random_state=42)
features = ['TRIP_MILES','TRIP_MINUTES']
heldout_model = make_pipeline(StandardScaler(), LinearRegression())
heldout_model.fit(training_df.iloc[train_i][features], training_df.iloc[train_i]['FARE'])
predictions = heldout_model.predict(training_df.iloc[test_i][features])
heldout_rmse = root_mean_squared_error(training_df.iloc[test_i]['FARE'], predictions)
print('Held-out RMSE (dollars):', heldout_rmse)
assert set(train_i).isdisjoint(test_i)
pd.DataFrame({'row':test_i,'observed':training_df.iloc[test_i]['FARE'].to_numpy(),
              'predicted':predictions}).to_csv(RESULTS/'taxi_heldout_predictions.csv',index=False)

## Save computational progress
Run the final cell after every exercise cell. It checks the required computational stages ran in this session, saves timestamped evidence, and synchronizes results/models to Drive. Save the notebook itself to retain its cell outputs. Website quiz/exercise progress is tracked separately by Google.


In [ ]:
finish_run(COLAB_CONTEXT)